# SHAP Explainability

**Covers:**
- Built-in feature importance (Random Forest)
- SHAP Summary Plot (global importance)
- SHAP Force Plots for:
  - True Positive (correctly caught fraud)
  - False Positive (legitimate flagged as fraud)
  - False Negative (missed fraud)
- Business recommendations

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib

from sklearn.metrics import confusion_matrix

shap.initjs()  # for inline JS plots in Jupyter

## 1. Load Model and Test Data

In [ ]:
rf = joblib.load('models/random_forest_fraud.joblib')

X_test = pd.read_csv('data/processed/X_test_fraud.csv')
y_test = pd.read_csv('data/processed/y_test_fraud.csv').squeeze()

import re
X_test.columns = [re.sub(r'[^\w]', '_', col) for col in X_test.columns]

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(f'Test set size: {len(X_test)}')

## 2. Built-in Feature Importance (Top 10)

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X_test.columns)
top10 = importance.nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
top10.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest Built-in Feature Importance (Top 10)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('data/processed/feature_importance.png', dpi=150)
plt.show()

## 3. SHAP Summary Plot (Global)

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# RandomForestClassifier: TreeExplainer returns SHAP values per class —
# select the values for class 1 (fraud)
if isinstance(shap_values, list):
    shap_values_fraud = shap_values[1]
    expected_value = explainer.expected_value[1]
elif shap_values.ndim == 3:
    shap_values_fraud = shap_values[:, :, 1]
    expected_value = explainer.expected_value[1]
else:
    shap_values_fraud = shap_values
    expected_value = explainer.expected_value

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_fraud, X_test, max_display=15, show=False)
plt.title('SHAP Summary Plot — Global Feature Importance')
plt.tight_layout()
plt.savefig('data/processed/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Identify Key Prediction Cases

In [ ]:
y_test_arr = y_test.values

# True Positive: actual fraud, predicted fraud
tp_idx = np.where((y_test_arr == 1) & (y_pred == 1))[0]
# False Positive: actual legitimate, predicted fraud
fp_idx = np.where((y_test_arr == 0) & (y_pred == 1))[0]
# False Negative: actual fraud, predicted legitimate
fn_idx = np.where((y_test_arr == 1) & (y_pred == 0))[0]

print(f'True Positives : {len(tp_idx)}')
print(f'False Positives: {len(fp_idx)}')
print(f'False Negatives: {len(fn_idx)}')

## 5. SHAP Force Plots

In [ ]:
i = tp_idx[0]
print(f'--- True Positive (index {i}) ---')
shap.force_plot(
    expected_value,
    shap_values_fraud[i],
    X_test.iloc[i],
    matplotlib=True
)
plt.title('Force Plot — True Positive (Fraud Correctly Caught)')
plt.tight_layout()
plt.savefig('data/processed/shap_force_tp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
i = fp_idx[0]
print(f'--- False Positive (index {i}) ---')
shap.force_plot(
    expected_value,
    shap_values_fraud[i],
    X_test.iloc[i],
    matplotlib=True
)
plt.title('Force Plot — False Positive (Legitimate Wrongly Flagged)')
plt.tight_layout()
plt.savefig('data/processed/shap_force_fp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
i = fn_idx[0]
print(f'--- False Negative (index {i}) ---')
shap.force_plot(
    expected_value,
    shap_values_fraud[i],
    X_test.iloc[i],
    matplotlib=True
)
plt.title('Force Plot — False Negative (Fraud Missed)')
plt.tight_layout()
plt.savefig('data/processed/shap_force_fn.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top 5 Fraud Drivers

Based on SHAP summary plot, fill in after running the analysis:

| Rank | Feature | Direction | Business Meaning |
|------|---------|-----------|------------------|
| 1 | `time_since_signup` | Low = fraud risk | Fraudsters act immediately after account creation |
| 2 | `purchase_value` | High = fraud risk | Large transactions are more likely fraudulent |
| 3 | `hour_of_day` | Late night = fraud risk | Fraud spikes at unusual hours |
| 4 | `user_tx_count` | High = fraud risk | Rapid repeated transactions signal fraud |
| 5 | `country_*` | Certain countries = risk | Geolocation correlates with fraud rates |


## 7. Business Recommendations

1. **New account velocity check:** Transactions within 1 hour of signup should trigger 2FA or manual review — `time_since_signup` is the strongest fraud signal.

2. **High-value transaction screening:** Automatically flag transactions above a threshold (e.g., top 5% of `purchase_value`) for additional verification when combined with other risk factors.

3. **Geographic risk scoring:** Assign real-time country risk scores based on fraud rates observed in training data. High-risk-country transactions should route through enhanced verification.

4. **Off-hours monitoring:** Increase alerting sensitivity for transactions during 1am–4am when fraud rate is highest (per `hour_of_day` analysis).

5. **Velocity limits:** Users making more than N transactions in a short window (`user_tx_count`) should have subsequent transactions held pending review.
